## Department of Education
# Masterlist of Schools 2020-2021

Data wrangling notebook for parsing basic education schools data from PDF.

In [ ]:
# Setting up dependencies and output display formatting

from typing import List

import pandas as pd
import tabula
from tqdm import tqdm

pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

In [2]:
# reading the PDF tables using tabula
dfs_candidate = tabula.read_pdf("../data/education/masterlist.pdf", pages="all")

if isinstance(dfs_candidate, list):
    dfs: List[pd.DataFrame] = dfs_candidate
else:
    assert ValueError, "Not a list."

Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'


In [3]:
# Joining all dfs to one root_df
root_df = pd.DataFrame()
for idx, d in enumerate(tqdm(dfs)):
    d["page"] = idx
    root_df = pd.concat([root_df, d])
root_df = root_df.reset_index(drop=True)


  0%|          | 0/544 [00:00<?, ?it/s]

100%|██████████| 544/544 [00:03<00:00, 147.54it/s] 


## Data Cleaning
### Column Values Inspection

In [4]:
root_df.sample(5)

,Region,Division,District,BEIS School ID,School Name,Street Address,Municipality,Legislative District,Barangay,Sector,Urban/Ru,Sacl hColaosls Sifuicbactliaosnsification,Modified Curricural Offering Classification,page
55951,CAR,Abra,Sallapadan,135184,Gangal Elementary School,N/A,SALLAPADAN,Lone District,GANGAL (POB.),Public,Partially U,bDaenpED Managed,Purely ES,499
20657,Region V,Albay,Camalig North,111637,Quirangay Elementary School,QUIRANGAY CAMALIG ALBAY,CAMALIG,2nd District,QUIRANGAY,Public,Partially U,bDaenpED Managed,Purely ES,184
32636,Region VII,Cebu,Tuburan I,119809,Daanlungsod ES,"Daaan Lungsod, Tuburan, Cebu",TUBURAN,3rd District,DAAN LUNGSOD,Public,Partially U,bDaenpED Managed,Purely ES,291
832,Region I,Ilocos Sur,Sta. Cruz,400049,"Sta. Cruz Institute, Inc.",N/A,SANTA CRUZ,2nd District,POBLACION SUR,Private,Partially U,bSeacntarian,All Offering (K to 12),7
40152,Region IX,Zamboanga del Sur,Bayog,124919,Depore ES,Purok 2,BAYOG,2nd District,DEPORE,Public,Partially U,bDaenpED Managed,Purely ES,358


I'm going to standardize columns to snake case and also correct column names along the
way. Sample erring name is `Sacl hColaosls Sifuicbactliaosnsification` 😁

In [5]:
# renaming to snake case because we like consistency
correct_column_names = {
    "Region": "region",
    "Division": "division",
    "District": "district",
    "BEIS School ID": "beis_school_id",
    "School Name": "school_name",
    "Street Address": "street_address",
    "Municipality": "municipality",
    "Legislative District": "legislative_district",
    "Barangay": "barangay",
    "Sector": "sector",
    "Urban/Ru": "settlement_type",
    "Sacl hColaosls Sifuicbactliaosnsification": "school_subclassification",
    "Modified Curricural Offering Classification": "modified_cultural_offering_classification",
    "page": "page",
}

root_df = root_df.rename(correct_column_names, axis=1)

#### Small Hurdle: Fixing Low-Cardinality Categories
There are a few low-cardinality categorical data in here that got messed up during the
parsing of the PDF. I'm going to investigate them one by one and check in the actual PDF
the correct value.

In [6]:
root_df["sector"].value_counts(dropna=False)
# looks clean, bet lets convert to snake case for consistency
# --> any categorical data, convert to snake_case

sector
Public       47421
Private      13256
SUCs/LUCs      247
Name: count, dtype: int64

In [7]:
root_df["sector"] = root_df["sector"].replace("Public", "public")
root_df["sector"] = root_df["sector"].replace("Private", "private")
root_df["sector"] = root_df["sector"].replace("SUCs/LUCs", "suc_luc")
root_df.sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
38308,Region VIII,Samar (Western Samar),Sta. Rita I,123766,Guinbalot-an ES,-,SANTA RITA,2nd District,GUINBALOT-AN,public,Partially U,bDaenpED Managed,Purely ES,342
60215,NCR,Las Piñas City,Las Piñas City I,407090,Brittany School of Las Pinas (JEIB Star Academ...,21 Aldebaran Zapote Las Pinas City,CITY OF LAS PIÑAS,Lone District,ZAPOTE,private,Urban,Non-Sectarian,All Offering (K to 12),537
11177,Region III,Tarlac City,Tarlac North,istrict A500389,Balibago Primero IS,Block 2,CITY OF TARLAC (Capital),2nd District,BALIBAGO I,public,Partially U,bDaenpED Managed,All Offering (K to 12),99
47150,Region XI,Davao City,Sta. Ana,409377,"Young David's Precious Lamb, Inc.","San Jose, Davencor, Brgy. Gov. Vicente Duterte...",DAVAO CITY,1st District,GOV. VICENTE DUTERTE,private,Partially U,bNaonn-Sectarian,Purely ES,421
7967,Region III,Nueva Ecija,Carranglan,105244,Digdig-Joson ES,NATIONAL HIGHWAY,CARRANGLAN,2nd District,JOSON (DIGIDIG),public,Partially U,bDaenpED Managed,Purely ES,71


In [8]:
root_df["settlement_type"].value_counts(dropna=False)
# looks clean also. but let's convert to snake_case

settlement_type
Partially U    47606
Urban          10404
Rural           2914
Name: count, dtype: int64

In [9]:
root_df["settlement_type"] = root_df["settlement_type"].replace(
    "Partially U", "partially_urban"
)
root_df["settlement_type"] = root_df["settlement_type"].replace("Urban", "urban")
root_df["settlement_type"] = root_df["settlement_type"].replace("Rural", "rural")
root_df.sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
16502,Region IV-A,San Pablo City,San Francisco,109795,San Antonio I Elementary School,Maharlika Highway,SAN PABLO CITY,3rd District,SAN ANTONIO 1,public,urban,DepED Managed,Purely ES,147
44106,Region X,Misamis Oriental,Opol,405172,Opol Adventist Elementary School,"Zone 6, Poblacion, Opol, Misamis Oriental",OPOL,2nd District,POBLACION,private,partially_urban,bSeacntarian,Purely ES,393
22568,Region V,Camarines Sur,Ragay,173544,Lower Sta Cruz ES,Lower Sta. Cruz,RAGAY,1st District,LOWER SANTA CRUZ,public,partially_urban,bDaenpED Managed,Purely ES,201
33212,Region VII,Mandaue City,Mandaue City,Central Distr1ic1t9996,Ibabao-Estancia ES,S.B. Cabahug St.,MANDAUE CITY,6th District,IBABAO-ESTANCIA,public,urban,DepED Managed,Purely ES,296
13091,Region IV-A,Cavite,Naic I,424263,Colegio De Montessori,Timalan Concepcion,NAIC,7th District,TIMALAN CONCEPCION,private,partially_urban,bNaonn-Sectarian,All Offering (K to 12),116


In [10]:
root_df["school_subclassification"].value_counts(dropna=False)
# a little dirty, gonna double check all values that are unreadable, double check in
# the actual PDF, and then create a rename dictionary for it

school_subclassification
bDaenpED Managed                 40457
DepED Managed                     6795
Non-Sectarian                     5244
bNaonn-Sectarian                  4252
bSeacntarian                      2595
Sectarian                         1163
bSUanC Managed                     160
bLoacnal Government                109
SUC Managed                         50
Local Government                    45
bLUanC                              21
LUC                                 16
bDaOnST Managed                      9
DOST Managed                         4
bLoacnal International School        2
bOatnher GA Managed                  1
Other GA Managed                     1
Name: count, dtype: int64

In [11]:
root_df[root_df["school_subclassification"] == "bSUanC Managed"].sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
20601,Region IV-B,Calapan City,Calapan West,600045,Mindoro State College of Agriculture and Techn...,Nautical Highway,CITY OF CALAPAN (Capital),1st District,MASIPIT,suc_luc,partially_urban,bSUanC Managed,JHS with SHS,183
10219,Region III,Zambales,Iba,600029,Ramon Magsaysay Technological University,"Iba, Zambales",IBA (Capital),2nd District,ZONE 6 POB. (BAYTAN),suc_luc,partially_urban,bSUanC Managed,JHS with SHS,91


In [12]:
root_df[root_df["school_subclassification"] == "bLoacnal Government"].sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
34063,Region VII,Toledo City,Toledo City W,st District190502,Sangi ES,Sangi Toledo City,TOLEDO CITY,3rd District,SANGI,public,partially_urban,bLoacnal Government,Purely ES,304
26491,Region VI,Capiz,Pilar,501100,Guise Integrated School,GUISE,PILAR,1st District,DULANGAN,public,partially_urban,bLoacnal Government,ES and JHS (K to 10),236


In [13]:
root_df[root_df["school_subclassification"] == "bDaOnST Managed"].sample(2)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
41553,Region IX,Dipolog City,Dipolog City E,st305575,Philippine Science High School - Zamboanga Pen...,Cogon,DIPOLOG CITY (Capital),2nd District,COGON,public,partially_urban,bDaOnST Managed,Purely JHS,371
50573,Region XII,Koronadal City,Koronadal We,t District I330521,Philippine Science High School - SOCCSKSARGEN ...,Not Applicable,CITY OF KORONADAL (Capital),2nd District,PARAISO,public,partially_urban,bDaOnST Managed,JHS with SHS,451


In [14]:
root_df[root_df["school_subclassification"] == "bLoacnal International School"].sample(
    2
)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
21401,Region V,Camarines Norte,Labo East,409757,"Camarines Norte International School, Inc.","Maharlika Highway, P-1",LABO,1st District,MASALONG,private,partially_urban,bLoacnal International School,Purely SHS,191
21400,Region V,Camarines Norte,Labo East,409756,"ADR Bicol International Technological College,...",P-4,LABO,1st District,MALASUGUI,private,partially_urban,bLoacnal International School,Purely SHS,191


In [15]:
root_df[root_df["school_subclassification"] == "bOatnher GA Managed"].sample()

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
6048,Region II,Tuguegarao City,Tuguegarao,est Educatio1n0a0l 9Z9o4,eDepartment of Agriculture R02 Child Developme...,"Nursery Compound, San Gabriel, Tuguegarao City",TUGUEGARAO CITY(Capital),3rd District,SAN GABRIEL,public,partially_urban,bOatnher GA Managed,Purely ES,54


In [16]:
# creating the replace dictionary
mapper = {
    "bDaenpED Managed": "deped_managed",
    "DepED Managed": "deped_managed",
    "Non-Sectarian": "non_sectarian",
    "bNaonn-Sectarian": "non_sectarian",
    "bSeacntarian": "sectarian",
    "Sectarian": "sectarian",
    "bSUanC Managed": "suc_managed",
    "bLoacnal Government": "local_government",
    "SUC Managed": "suc_managed",
    "Local Government": "local_government",
    "bLUanC": "luc",
    "LUC": "luc",
    "bDaOnST Managed": "dost_managed",
    "DOST Managed": "dost_managed",
    "bLoacnal International School": "local_international_school",
    "bOatnher GA Managed": "other_ga_managed",
    "Other GA Managed": "other_ga_managed",
}
# nice list 😉

In [17]:
# now im gonna replace
for key, value in mapper.items():
    root_df["school_subclassification"] = root_df["school_subclassification"].replace(
        key, value
    )
root_df.sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
28121,Region VI,Iloilo,San Dionisio,302541,Nicomedes R. Tubar Sr. NHS,Brgy. Pase,SAN DIONISIO,5th District,PASE,public,partially_urban,deped_managed,JHS with SHS,251
52355,CARAGA,Surigao del Sur,Lianga,132773,Ganayon Elementary School,"Ganayon, Lianga, Surigao del Sur",LIANGA,1st District,GANAYON,public,partially_urban,deped_managed,Purely ES,467
39347,Region IX,Zamboanga del Norte,Gutalac II,124316,New Dapitan ES,-,GUTALAC,3rd District,NEW DAPITAN,public,partially_urban,deped_managed,Purely ES,351
48157,Region XII,North Cotabato,Kabacan Sout,304444,Kabacan National High School,Mapanao Street,KABACAN,1st District,POBLACION,public,partially_urban,deped_managed,JHS with SHS,430
30689,Region VII,Bohol,Getafe,118334,Jandayan ES,Jandayan Norte,GETAFE,2nd District,JANDAYAN NORTE,public,rural,deped_managed,Purely ES,274


In [18]:
root_df["modified_cultural_offering_classification"].value_counts()
# looks clean => convert to snake case

modified_cultural_offering_classification
Purely ES                 43765
JHS with SHS               7490
All Offering (K to 12)     3423
ES and JHS (K to 10)       3056
Purely JHS                 1787
Purely SHS                 1403
Name: count, dtype: int64

In [19]:
offering_mapper = {
    "Purely ES": "purely_es",
    "JHS with SHS": "jhs_with_shs",
    "All Offering (K to 12)": "all_offering",
    "ES and JHS (K to 10)": "es_and_jhs",
    "Purely JHS": "purely_jhs",
    "Purely SHS": "purely_shs",
}
root_df["modified_cultural_offering_classification"] = root_df[
    "modified_cultural_offering_classification"
].replace(offering_mapper)

root_df.sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page
11646,Region IV-A,Dasmarinas City,Dasmariñas I,301187,Pag-asa NHS (Dasmarinas NHS - Pag-asa Annex),Victoria Reyes,CITY OF DASMARIÑAS,4th District,VICTORIA REYES,public,urban,deped_managed,purely_jhs,104
50462,Region XII,Kidapawan City,Kidapawan Cit,District II131338,Sayaban Elementary School,Sayaban,CITY OF KIDAPAWAN (Capital),2nd District,ILOMAVIS,public,partially_urban,deped_managed,purely_es,450
37512,Region VIII,Northern Samar,Catarman I,122857,Catarman SPED Center,"A. Mabini St., Brgy. Acacia, Catarman Northern...",CATARMAN (Capital),1st District,ACACIA (POB.),public,partially_urban,deped_managed,purely_es,334
29498,Region VI,La Carlota City,La Carlota City,District III117648,La Granja ES,Brgy. La Granja,LA CARLOTA CITY,4th District,LA GRANJA,public,partially_urban,deped_managed,purely_es,263
7642,Region III,Bulacan,San Miguel So,th105106,Paliwasan ES,-,SAN MIGUEL,3rd District,PALIWASAN,public,partially_urban,deped_managed,purely_es,68


#### Medium Hurdle: Fixing High-Cardinality BEIS School ID

A medium-level hurdle: the BEIS School ID is not parsed correctly. And this one has high
cardinality so using usual fix and replace will not work, at least not on scale that I
want. Let's use other methods

In [20]:
# my guess is that there's a set of patterns which I can use to transform the dirty data
# to clean data. I'm going to accumulate the data I've cleaned so far in this list
list_of_clean_dfs: List[pd.DataFrame] = []

In [21]:
# checking for columns I can change to numeric values. but for that I need a scratch
# column to work with so we can type cast all I want. Gonna name it beis_1
root_df["beis_1"] = pd.to_numeric(root_df["beis_school_id"], errors="coerce").astype(
    "Int32"
)

In [ ]:
# Then I'm gonna isolate all columns that are 6 digits and are numerical (i.e. not null)
six_dig_and_not_null = (root_df["beis_1"].astype(str).str.len() == 6) & (
    root_df["beis_1"].notna()
)
root_df[six_dig_and_not_null].sample()

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
154,Region I,Ilocos Norte,Dingras II,100133,Mandaloque ES,Mandaloque,DINGRAS,2nd District,MANDALOQUE,public,partially_urban,deped_managed,purely_es,1,100133


In [23]:
# the six_dig_and_not_null ones are the clean ones so i'm gonna add it to the list
list_of_clean_dfs.append(root_df[six_dig_and_not_null])

In [24]:
# remove it from the root_df and we get the remaining data to be cleaned.
# from here im gonna name my df variables as "wdf" for working dataframe.
# i'll progressively add w once we isolate each type of transformation
wdf = root_df[~six_dig_and_not_null]
wdf.sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
59178,NCR,Caloocan City,Aromar (Aro,han-Marula1s3)6611,Libis Talisay ES,Gen. Luna St.,KALOOKAN CITY,2nd District,BARANGAY 10,public,urban,deped_managed,purely_es,528,<NA>
54686,BARMM,Maguindanao I,Datu Saudi Uy,Ampatuan216529,Elian PS,-,DATU SAUDI-AMPATUAN,2nd District,ELIAN,public,partially_urban,deped_managed,purely_es,488,<NA>
60503,NCR,Malabon City,Malabon Distr,ct I305447,Malabon National High School,M. Naval Street,CITY OF MALABON,Lone District,HULONG DUHAT,public,urban,deped_managed,jhs_with_shs,540,<NA>
6120,Region II,Cauayan City,Cauayan Nort,District103219,Rizal ES,"BRGY. RIZAL CAUAYAN CITY, ISA.",CITY OF CAUAYAN,3rd District,RIZAL,public,partially_urban,deped_managed,purely_es,54,<NA>
59917,NCR,Pasig City,Pasig City Dist,ict III136730,Francisco Legaspi Memorial School,F. Legaspi,CITY OF PASIG,Lone District,UGONG,public,urban,deped_managed,purely_es,535,<NA>


I noticed that for the remaining data, we can find, for example, `t II403488`

We can detect the succession of 6 digits using regular expressions (regex)

In [25]:
# I'm now trying that
wdf["beis_1"] = wdf["beis_school_id"].str.findall(r"\d+").str[0]
wdf.sample(3)

/tmp/ipykernel_29463/1976253110.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wdf["beis_1"] = wdf["beis_school_id"].str.findall(r"\d+").str[0]


,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
10763,Region III,Olongapo City,Olongapo City,District II422022,"The Manila Times College of Subic, Inc.","Hospital Compound, Upper Cubi Point SBFZ",OLONGAPO CITY,1st District,NaN,private,urban,non_sectarian,all_offering,96,422022
53207,BARMM,Maguindanao II,Sultan Kudara,I216558,Senditan PS,Senditan,SULTAN KUDARAT (NULING),1st District,SENDITAN,public,partially_urban,deped_managed,purely_es,475,216558
11460,Region III,Meycauayan City,Meycauayan,ast104920,Iba ES,ANA MARIA VILLAGE,CITY OF MEYCAUAYAN,4th District,IBA,public,urban,deped_managed,purely_es,102,104920


In [26]:
# I also noticed there are data with 7 digits. But that's likely wrong. Let's just keep
# the ones that have 6 digits
is_six_digit = (
    pd.to_numeric(wdf["beis_1"], errors="coerce").astype("Int32").astype(str).str.len()
    == 6
)
wdf[is_six_digit].sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
47305,Region XI,Digos City,Digos Occiden,al129754,Lungag ES,-,CITY OF DIGOS (Capital),1st District,LUNGAG,public,partially_urban,deped_managed,purely_es,422,129754
16178,Region IV-A,Lipa City,Lipa City East,istrict427039,"Mary Immaculate Montessori of Lipa City, Inc.","Camia St., City Park Subd., Sabang, Lipa City",LIPA CITY,4th District,SABANG,private,urban,non_sectarian,purely_es,144,427039
34172,Region VII,Danao City,Danao City Ea,t404351,"Sto.Tomas College,Danao City,Inc.","Bonifacio St., Brgy. Poblacion, Danao City",DANAO CITY,5th District,POBLACION,private,partially_urban,sectarian,jhs_with_shs,305,404351
2856,Region I,Dagupan City,Dagupan City,istrict I102149,Juan L. Siapno ES,Lasip Chico,DAGUPAN CITY,4th District,LASIP CHICO,public,urban,deped_managed,purely_es,25,102149
22259,Region V,Camarines Sur,Magarao-Bom,on500157,Ponong Integrated School,Zone 4,MAGARAO,3rd District,PONONG,public,partially_urban,deped_managed,all_offering,198,500157


In [27]:
# i'm now going to add that to the list of clean dfs
list_of_clean_dfs.append(wdf[is_six_digit])

In [28]:
# and on to the next wdf: wwdf
wwdf = wdf[~is_six_digit]
wwdf.sample(3)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
50278,Region XII,General Santos City,General Santo,City North 4D7is0t0ri1c6t,Creative Achiever Special Education (SPED) Aca...,"Aparente st., Brgy. City Heights, General Sant...",GENERAL SANTOS CITY (DADIANGAS),1st District,CITY HEIGHTS,private,partially_urban,non_sectarian,purely_es,448,4
33177,Region VII,Lapu-Lapu City,Lapu-Lapu Cit,South Distr3ic1t2704,Marigondon NHS-Sudtonggan Night High School,"Sudtonggan, Lapu-lapu City",LAPU-LAPU CITY (OPON),6th District,BASAK,public,urban,deped_managed,purely_jhs,296,3
44644,Region X,Gingoog City,Gingoog City,orth-3 Distr5ic0t0800,Dukdokaan Integrated School - Elem,"P6, Dukdokaan Eureka",GINGOOG CITY,1st District,EUREKA,public,partially_urban,deped_managed,es_and_jhs,398,3


We can now observe that new cases are now looking like `Central Distr4ic0t9876`
Which contains data `409876` but with characters mistakenly added in the middle.

In [29]:
# im going to use regex again
wwdf["beis_1"] = (
    pd.to_numeric(
        wwdf["beis_school_id"].str.findall(r"\d+").str.join(""), errors="coerce"
    )
    .astype("Int32")
    .astype(str)
)

# and detect which ones are 6 digits
nuther_six_dig = wwdf["beis_1"].str.len() == 6

/tmp/ipykernel_29463/1118215701.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wwdf["beis_1"] = (


In [30]:
# that's now considered as clean
list_of_clean_dfs.append(wwdf[nuther_six_dig])

In [31]:
# im going to name the next df as rdf. no reason behind it. i just want to.
rdf = wwdf[~nuther_six_dig]
rdf.sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
44564,Region X,Gingoog City,Gingoog City,ast-1 District128013,Lawit Elementary school,Purok #1 Lawit,GINGOOG CITY,1st District,LAWIT,public,partially_urban,deped_managed,purely_es,397,1128013
44562,Region X,Gingoog City,Gingoog City,ast-1 District128011,Hindangon Elementary School,Purok #4 Hindangon,GINGOOG CITY,1st District,HINDANGON,public,partially_urban,deped_managed,purely_es,397,1128011
32711,Region VII,Cebu,Pinamungajan,2119565,Pandacan Elementary School,"Pandacan, Pinamungajan, Cebu",PINAMUNGAHAN,3rd District,PANDACAN,public,partially_urban,deped_managed,purely_es,292,<NA>
44601,Region X,Gingoog City,Gingoog City S,uth-1 Distr1ic2t8057,Ricoro Elementary School,Purok #2 Ricoro,GINGOOG CITY,1st District,RICORO,public,partially_urban,deped_managed,purely_es,398,1128057
44600,Region X,Gingoog City,Gingoog City S,uth-1 Distr1ic2t8056,Ramon Arevalo Elementary School,Purok #1 Samay,GINGOOG CITY,1st District,SAMAY,public,partially_urban,deped_managed,purely_es,398,1128056


Hmm, these are 7 digits. I did some manual checks and most of them are due to the
district column also being mistakenly added in the beis_school_id column...
_at the front_. So let us remove that.

In [32]:
sev_dig = rdf["beis_1"].str.len() == 7
sev_df = rdf[sev_dig]
sev_df["beis_1"] = sev_df["beis_1"].str[1:] # removing the first char

/tmp/ipykernel_29463/1770584804.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sev_df["beis_1"] = sev_df["beis_1"].str[1:] # removing the first char


In [33]:
# that's now clean
list_of_clean_dfs.append(sev_df)

In [34]:
# gonna name the next df as wherf because y not (my stamina is now drained. im tired)
wherf = rdf[~sev_dig]
wherf.sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
32330,Region VII,Cebu,Pinamungajan,1303078,Lut-od NHS,"Lut-od, Pinamungajan, Cebu",PINAMUNGAHAN,3rd District,LUT-OD,public,partially_urban,deped_managed,jhs_with_shs,288,<NA>
32312,Region VII,Cebu,Pinamungajan,1119554,Binabag ES,"Kaipilan, Binabag, Pinamungajan",PINAMUNGAHAN,3rd District,BINABAG,public,partially_urban,deped_managed,purely_es,288,<NA>
32715,Region VII,Cebu,Pinamungajan,2119572,Tajao Central School,South,PINAMUNGAHAN,3rd District,TAJAO,public,partially_urban,deped_managed,purely_es,292,<NA>
32324,Region VII,Cebu,Pinamungajan,1119575,Tupas ES,-,PINAMUNGAHAN,3rd District,TUPAS,public,partially_urban,deped_managed,purely_es,288,<NA>
32317,Region VII,Cebu,Pinamungajan,1119562,Lut-od ES,"Lut-od, Pinamungajan, Cebu",PINAMUNGAHAN,3rd District,LUT-OD,public,partially_urban,deped_managed,purely_es,288,<NA>


In [35]:
# but these are just 7 digits... so i'm just going to remove the first digit as well
wherf["beis_1"] = wherf["beis_school_id"].astype(str).str[1:]

/tmp/ipykernel_29463/958271942.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wherf["beis_1"] = wherf["beis_school_id"].astype(str).str[1:]


In [38]:
wherf["beis_1"].str.len().value_counts()

beis_1
6    36
Name: count, dtype: int64

In [36]:
# adding it to the list...
list_of_clean_dfs.append(wherf)

Ahahahaha! All values are now cleaned... That concludes the BEIS transformation set.

I'm now going to concatenate them all into one dataset, which will be our base set.

In [39]:
mother_df: pd.DataFrame = pd.DataFrame()
for a_df in list_of_clean_dfs:
    mother_df = pd.concat([mother_df, a_df])

In [41]:
# checking our old beis_school_id len distribution
mother_df["beis_school_id"].astype(str).str.len().value_counts()

beis_school_id
6     45719
7      2228
8      2122
9      1761
16      985
11      949
10      909
18      877
13      813
14      676
15      619
19      581
12      570
20      490
17      428
21      370
22      241
27      150
24      132
26       91
23       85
25       85
28       43
Name: count, dtype: int64

In [42]:
# replacing the dirty beis_school_id with our working beis_1
mother_df["beis_school_id"] = mother_df["beis_1"]

In [43]:
# and now checking again
mother_df["beis_school_id"].astype(str).str.len().value_counts()

beis_school_id
6    60924
Name: count, dtype: int64

Nice, all are standard now.

Checking how it looks...

In [ ]:
mother_df.sample(10)
# ... real nice

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,page,beis_1
8509,Region III,Nueva Ecija,San Isidro,401047,J. Victoria Montessori School,Rizal St.,SAN ISIDRO,4th District,MALAPIT,private,partially_urban,non_sectarian,all_offering,76,401047
13260,Region IV-A,Cavite,Silang II,424166,"Madre Giuditta Martelli School, Inc.","Lalaan II, Silang, Cavite",SILANG,5th District,LALAAN II,private,partially_urban,sectarian,purely_es,118,424166
22384,Region V,Camarines Sur,Ocampo,112862,San Jose Oras Elementary School,n/a,OCAMPO,3rd District,SAN JOSE ORAS,public,partially_urban,deped_managed,purely_es,199,112862
56673,CAR,Benguet,La Trinidad,479517,GLOBALIGHT VISION SCHOOL,"Ma-e, Bahong, La Trinidad, Benguet",LA TRINIDAD (Capital),Lone District,CRUZ,private,partially_urban,non_sectarian,purely_es,506,479517
1525,Region I,"Pangasinan I, Lingayen",Bani,101239,Luac ES,"-Luac, Bani, Pangasinan",BANI,1st District,LUAC,public,partially_urban,deped_managed,purely_es,13,101239
24334,Region V,Sorsogon,Matnog,114307,Cabagahan ES,"Cabagahan, Matnog, Sor.",MATNOG,2nd District,CABAGAHAN,public,partially_urban,deped_managed,purely_es,217,114307
8958,Region III,Pampanga,Guagua East,105999,San Juan Nepomuceno ES,PUROK 4,GUAGUA,2nd District,SAN JUAN NEPOMUCENO,public,urban,deped_managed,purely_es,80,105999
55871,CAR,Abra,Lagangilang,135057,Caridad Azares ES,n/a,LAGANGILANG,Lone District,NAGTUPACAN,public,partially_urban,deped_managed,purely_es,498,135057
48820,Region XII,North Cotabato,Pigcawayan W,130290,Libungan Torreta Elementary School,"Libungan Torreta, Pigcawayan, Cotabato",PIGKAWAYAN,1st District,LIBUNGAN TORRETA,public,partially_urban,deped_managed,purely_es,435,130290
55034,BARMM,Sulu,Talipao,134643,Pang. Karimuddin Elementary School,Lagtoh Talipao,TALIPAO,1st District,LAGTOH,public,partially_urban,deped_managed,purely_es,491,134643


In [46]:
# some few housekeeping:

# because PDF people don't start counting from 0 but we do 😉
mother_df["page"] = mother_df["page"] + 1 

# for more self-documenting dataset
mother_df = mother_df.rename({"page": "masterlist_page"}, axis=1)

# dropping our scratch column
mother_df = mother_df.drop("beis_1", axis=1)

In [47]:
# final checks
mother_df.sample(5)

,region,division,district,beis_school_id,school_name,street_address,municipality,legislative_district,barangay,sector,settlement_type,school_subclassification,modified_cultural_offering_classification,masterlist_page
12351,Region IV-A,Batangas,San Jose,107590,Dagatan Elementary School,"Dagatan, San Jose, Batangas",SAN JOSE,4th District,DAGATAN,public,partially_urban,deped_managed,purely_es,111
38681,Region VIII,Calbayog City,Calbayog Distr,124012,Giragaan Elementary School,Purok-1,CALBAYOG CITY,1st District,GERAGA-AN,public,partially_urban,deped_managed,purely_es,346
27009,Region VI,Iloilo,San Joaquin S,501456,Igcores Integrated School,Brgy. Igcores,SAN JOAQUIN,1st District,IGCORES,public,partially_urban,deped_managed,es_and_jhs,242
11507,Region III,Meycauayan City,Meycauayan,500439,Bancal Integrated School,"San Diego Street, Bancal",CITY OF MEYCAUAYAN,4th District,BANCAL,public,urban,deped_managed,es_and_jhs,103
19023,Region IV-B,Oriental Mindoro,Pola,110597,Matulatula ES,"Matulatula Proper, Pola, Oriental Mindoro",POLA,1st District,MATULATULA,public,partially_urban,deped_managed,purely_es,170


In [ ]:
# looks good... we can now save to parquet
mother_df.to_parquet(
    "../data/education/basic_education_institutions.parquet", index=False
)

#### __✨ALL DONE✨__

#### __Continuation: Hard Hurdle__
I'll be matching Philippine Standard Geographic Code ID  (PSGC 10-digit ID) to this
dataset. That means I have to match the location data, (region, province, municipality,
barangay) found from this dataset against the PSGC one. This is high-cardinality
matching without fixed set of transformations like we had above. This time we need a
slightly more complicated approach, which is ___fuzzy matching___.

